<div style="text-align: center;" >
<h1 style="margin-top: 0.2em; margin-bottom: 0.1em;">Runtime Analysis and Parallelism</h1>
<h4 style="margin-top: 0.7em; margin-bottom: 0.3em; font-style:italic"></h4>
</div>
<br>




# Google Slides
[Link](https://docs.google.com/presentation/d/1z47qngNk2k6xq-LMrWGKYIH-NR-Y6OZuNxk3GH-FeqE/edit?usp=sharing)

### __1. Evaluating Runtime__

**Effectivity of Resources**
- Space Usage:
   - Evaluate the space complexity of your algorithm, though it can be challenging with the standard library. Monitor for runtime errors such as RuntimeError, MemoryError, IOError, etc.


- Run Time:

  - Track the runtime of your algorithm using the [time module](https://docs.python.org/3/library/time.html). Use various time-related functions to profile runtime.




In [ ]:
import time

def factorial(n):
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result

start_time = time.time()

result = factorial(1000)

end_time = time.time()

print(f"The factorial of 10 is: {result}")
print(f"Execution time: {end_time - start_time} seconds")


In [ ]:

def factorial(n):
    result = 1
    for i in range(1, n + 1):
        time.sleep(1)
        result *= i
    return result

start_time = time.time()

result = factorial(5)

end_time = time.time()

print(f"The factorial of 10 is: {result}")
print(f"Execution time: {end_time - start_time} seconds")


You can also track the runtime using the [tqdm](https://tqdm.github.io/) package. 

In [ ]:
from tqdm import tqdm

In [ ]:
for i in tqdm(range(10000000)):
    pass

In [ ]:
import pandas as pd
import random

# Generate random data
names = ["Alice", "Bob", "Charlie", "David", "Emma", "Frank", "Grace", "Henry", "Ivy", "Jack",
         "Katherine", "Leo", "Mia", "Nathan", "Olivia", "Peter", "Quinn", "Rose", "Sam", "Taylor"]

ages = [random.randint(18, 30) for _ in range(20)]
studies = ["Computer Science", "Mathematics", "Physics", "Engineering", "Biology",
           "Psychology", "History", "Economics", "English", "Art History",
           "Chemistry", "Political Science", "Sociology", "Music", "Geology",
           "Philosophy", "Communication", "Anthropology", "Statistics", "Foreign Languages"]

# Create DataFrame
df = pd.DataFrame({
    'name': random.sample(names, 20),
    'age': ages,
    'study': random.choices(studies, k=20)
})

# Display the DataFrame
df

In [ ]:
for col, row in tqdm(df[:10].iterrows(), total=df[:10].shape[0]):
    time.sleep(0.5)
    print(f"{row['name']} is {row['age']} years old and is currently studying {row['study']}")

### __2. Parallelism__

In [ ]:
import time               # used to track run times
import threading          # used for multi threading
import os                 # used to check for available computing cores
import requests           # used for 'standard' web scraping
import aiohttp            # used for multi threaded web scraping
import asyncio            # used for writing async functions
import json               # work with json files
import multiprocessing    # used for multi processing

***Note: Depending on the IDE you use asyncio works a bit different. It has to do with something called 'event loop'. Check out this [post](https://stackoverflow.com/questions/55409641/asyncio-run-cannot-be-called-from-a-running-event-loop-when-using-jupyter-no) for more info regarding asyncio and jupyter.***

***Note: Depending on your operating system (OS) multi processing in python works differently. This is due to some fundamental differences in how the different operating systems deal with multi processing. If you want to get some more info on the differences between Windows/Mac and Linux check out this [resource](https://pythonforthelab.com/blog/differences-between-multiprocessing-windows-and-linux/).***

### 2.1. Concurrency vs. Parallelism

Often times it is not really noticeable for us humans whether something is run concurrently or in parallel. We're just to slow...<br>
In simple terms, 'concurrency' means that different parts of a greater whole can be run out of order while still producing the same result. 'Parallelism' means that different parts of a greater whole are run at the exact same time in, well..., parallel.

### Multi Threading - Concurrency

Multiple threads are run after each other.<br>
This happens so fast, though, that you might easily assume that they are run in parallel. A good example for this are Input/Output (IO) tasks where the different processes are (mostly) independent of each other. If you want to retrieve the content of a web page for example you will have to query the server for the information you need (the input). The server will then return the answer or information (the output) to you. While you are waiting for the server to respond and give you the output your machine is basically having a grand ol' time chilling and relaxing since there is nothing else to do than wait. Now, we can't have that of course! Every free minute needs to be invested into work, work, WORK...!<br>
So... Instead of just letting our machine lounge around, we tell it to send another query to another web page while it is waiting for the first web page to answer. And while it is still waiting we tell it to send another query, and another, and another... until our machine wishes there were something like machine rights or a labor union for computers.<br>
I hope you kinda get the point. We are not doing this stuff at the very same time but rather one step after the other. Since it only takes our machine a split second to send out a new query, though, we might get the impression that everything is working in parallel. Once the answers of the web pages are returned, they are for example stored and then printed.<br>

Think of it like this:<br>
You open your favorite browser and decide to watch a movie on the platform of your choice. So you enter in the address of the chosen web page. After you hit 'search' you realize that you do not know which movie you wanna watch. So you open a second tab and look for the best movies of your chosen genre while the first tab opening your streaming service is still loading. You decide on a movie and once you go back to your fist tab, it's finished loading and your good to go!

<div style="text-align: center;" >
<img src="sources/multi_threading.png" alt="Multi Threading Vis">
</div>

### Basic Example

To start things of we'll just have a look at a simple function that counts down from a specified integer number. First we do this without any 'fancy pants stuff' and afterwards we'll see how a multi threading approach would look like.

In [ ]:
count = 50_000_000

def countdown(n):
    while n > 0:
        n -= 1

In [ ]:
# Without multi threading
start = time.time()
countdown(count)
end = time.time()

print('Seconds:', end - start)

If we were to perform the same task using multi threading would the run time be:<br>
A: faster<br>
B: the same<br>
C: slower<br>
D: I don't know<br>

In [ ]:
# With multi threading
t1 = threading.Thread(target=countdown, args=(count//3,))
t2 = threading.Thread(target=countdown, args=(count//3,))
t3 = threading.Thread(target=countdown, args=(count//3,))

start = time.time()
t1.start()
t2.start()
t3.start()
t1.join()
t2.join()
t3.join()
end = time.time()

print('Seconds:', end - start)

Well, remember when I explained that good examples for multi threaded solutions are IO tasks where the different processes are (mostly) independent of each other? That is exactly what causes this behavior.<br>
Creating and managing the different threads costs some time. The 'normal' solution uses a single thread and just deducts 1 every iteration. This is basically the same as using three threads that also iteratively deduct 1. Since the threads are not running at the same time but rather each after the other (t1, t2, t3, t1, t2, t3,...) we do not see an improved run time. Because of the thread management we actually observe a slightly slower run time.

### Web Requests

For this part we are gonna deal with something a bit more advanced. The code below is copied and adapted from the [`aiohttp`](https://docs.aiohttp.org/en/stable/index.html) libraries documentation. First, let's try our 'standard' approach though:

In [ ]:
websites = ['https://en.wikipedia.org/', 'https://www.python.org/', 'https://stackoverflow.com/', 'https://stackexchange.com/', 'https://www.uni-konstanz.de/',
            'https://en.wikipedia.org/', 'https://www.python.org/', 'https://stackoverflow.com/', 'https://stackexchange.com/', 'https://www.uni-konstanz.de/',
            'https://en.wikipedia.org/', 'https://www.python.org/', 'https://stackoverflow.com/', 'https://stackexchange.com/', 'https://www.uni-konstanz.de/']

In [ ]:
# Without multi threading
start = time.time()
for site in websites:
    request = requests.get(site)
    print(request)
    html = request.text
    print('First char of website html:', html[0])

end = time.time()

print('Seconds:', end - start)

If we were to perform the same task using multi threading would the run time be:<br>
A: faster<br>
B: the same<br>
C: slower<br>
D: I don't know<br>

In [ ]:
# With multi threading
async def main(url):

    async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:

            print('Status:', response.status)
            html = await response.text()
            print('First char of website html:', html[0])

start = time.time()
for site in websites:
    await main(site) # if you do not use jupyter you will need to use 'asyncio.run(main())' instead

end = time.time()

print('Seconds:', end - start)

There is still room to adapt and further maximize the efficiency of this 'web scrape' so feel free to play around and try to maximize your performance boost.<br>

### Race Condition

A race condition is a situation in which multiple instructions are executed at the same time. If the different instructions finish faster/slower than expected this can lead to problems. Have a look at the following code and then try to answer the question down below.

What is the final result of x, if we run the cell below:<br>
A: 20<br>
B: 25<br>
C: None (we encounter an error)<br>
D: I don't know<br>

In [ ]:
x = 10
 
def increment(n):
    global x
 
    local_counter = x
    local_counter += n
 
    time.sleep(1)
 
    x = local_counter
    print(f'{threading.current_thread().name} increments x by {n} -> x: {x}\n')

# creating threads
t1 = threading.Thread(target=increment, args=(5,))
t2 = threading.Thread(target=increment, args=(10,))
 
# starting the threads
t1.start()

t2.start()
 
# waiting for the threads to complete
t1.join()
t2.join()
 
print(f'The final value of x is {x}')

This is not really what we want though. We want x to first be increased by 5 and then again by 10.<br>
To remedy the bug apparent in the above code we can use a [`threading.Lock`](https://docs.python.org/3/library/threading.html) which prevents our code from skipping the 1 second sleep and therefore ensures that the final result is correct. The `threading.Lock()`object gets passed to the function `increment`, wherein the `.aquire()`method is called. This makes sure that another thread cannot race by our current thread until the `.release()`method is called. Because of this we first increment x by 5 and then by 10 resulting in 25.<br>
If you code yourselves you need to keep in mind that sometimes multi threading can cause race conditions and they in turn may cause bugs. So be wary of the order your multi threading approaches are executed in!

In [ ]:
x = 10
 
def increment(n, lock):
    global x
 
    lock.acquire()
    local_counter = x
    local_counter += n
 
    time.sleep(1)
 
    x = local_counter
    
    lock.release()
    print(f'{threading.current_thread().name} increments x by {n} -> x: {x}')

# creating lock
lock = threading.Lock()

# creating threads
t1 = threading.Thread(target=increment, args=(5, lock))
t2 = threading.Thread(target=increment, args=(10, lock))
 
# starting the threads
t1.start()
t2.start()
 
# waiting for the threads to complete
t1.join()
t2.join()
 
print(f'\nThe final value of x is {x}')

### 2.2. Multi Processing - Parallelism

Multiple cores, capable of running multiple threads, are running at the exactly same time.<br>
This happens so fast, though, that you might easily assume that they are run in parallel and well, yes, they are! Good examples for this are processes that are a bit more taxing (or/and) can be split among multiple cores. Think of running a classification model for example. The goal is to classify a huge amount of data according to some rule set and the data can easily be broken into smaller subsets.<br>
You simply provide multiple cores with the same instructions but slightly different data aka different subsets of your original data. Then each core starts classifying their respective subset of the data and once done you can collect the results from all cores and put them together. The beautiful part about this is, that you can also provide different instructions to different cores while providing all of your cores with the same data.<br>

Think of it like this:<br>
You have to calculators and the motor skills to use them both at the exact same time.<br>
In one scenario you have a list of values (your data) and you want to perform some mathematical operation on those values. In order to halve the time needed to perform this operation split the list of values in two and use one calculator to run the operations on the first halve and the other calculator to run the operations on the second halve.<br>
In the other scenario you have a list of values (your data) and you want to perform two different mathematical operations, A and B, on those values. So, naturally, you use one calculator to run operation A on all values and the other calculator to run operation B on all values. Thereby halving the time needed to complete the task.

<div style="text-align: center;" >
<img src="sources/multi_processing.png" alt="Multi Processing Vis">
</div>

### Basic Example

We're gonna look at the same example as above. First without any multi blabla and then with a multi processing approach.

In [ ]:
# Without multi processing
start = time.time()
countdown(count)
end = time.time()

print('Seconds:', end - start)

If we were to perform the same task using multi processing would the run time be:<br>
A: faster<br>
B: the same<br>
C: slower<br>
D: I don't know<br>

In [ ]:
print('Available cores:', os.cpu_count())

# With multi processing 
p1 = multiprocessing.Process(target=countdown, args=(count//3,))
p2 = multiprocessing.Process(target=countdown, args=(count//3,))
p3 = multiprocessing.Process(target=countdown, args=(count//3,))

start = time.time()
p1.start()
p2.start()
p3.start()
p1.join()
p2.join()
p3.join()
end = time.time()

print('Seconds:', end - start)

A third of our countdown, respectively, is performed on one of three parallelly running cores which effectively reduces the run time to about a third. Some time is required to fire up the processes on three different cores but in the present example this is close to 0.